# Does explicit state help at all?
Hypothesis: Explicit state reduces repeated errors and improves solve rate compared to no state.

Task: Connections puzzles
- 16 words
- 4 groups of 4

Compare 4 things
1. One-shot: Model just solves directly
2. Chain-of-thought: Model reasons step-by-step in text
3. Agentic (ChatGPT / Claude)
4. State system

**Does explicit state help at all?**
- Why?
- Why not?

---

## Why Connections?
Connections is an evaluation that reduces pure memorization. It is also strong because it tests, elimination, constraint satisfaction, multi-step reasoning and error recovery. 

---



## One shot
Single model call, no iteration, no feedback, no retries

Input:
16 words
Output:
4 groups of 4
```{markdown}
You are solving a Connections-style puzzle.

You are given 16 words. Your task is to group them into 4 groups of 4 words based on a shared theme.

Return exactly 4 groups of 4 words.

Words:
{WORDS}

Output format:
Group 1: word1, word2, word3, word4
Group 2: ...
Group 3: ...
Group 4: ...
```
This tests pure capability without structure. We expect that this cannot recovery from mistakes and also high error rate. 

Good — you’re doing it right. I’ll fill the other three cleanly so they’re **comparable and not messy**.

---

## **CoT (Chain-of-Thought baseline)**

### Definition

Single call, but model is allowed to reason **in text before answering**

* still **no iteration**
* no external state
* no feedback loop

---

### Prompt

```{markdown}
You are solving a Connections-style puzzle.

You are given 16 words. Your task is to group them into 4 groups of 4 words based on a shared theme.

Think step by step. Identify possible groupings and eliminate incorrect ones before giving your final answer.

Words:
{WORDS}

Final answer format:
Group 1: word1, word2, word3, word4
Group 2: ...
Group 3: ...
Group 4: ...
```

---

### What this tests

Implicit reasoning inside the model

* grouping
* elimination
* internal “state” (but hidden in text)

---

### Weakness

* cannot enforce consistency
* may contradict earlier reasoning
* cannot track rejected groups reliably

---

## Agentic baseline

### Definition

Multi-step interaction with the model

* multiple calls
* model can revise answers
* may self-correct

---

### Setup

Loop:

1. propose groups
2. check correctness (external or simulated)
3. give feedback:
   * “this group is incorrect”
   * “you already tried this”
4. model tries again

---

### Prompt (core idea)

```{markdown}
You are solving a Connections-style puzzle.

You will propose groups of 4 words. After each attempt, you will receive feedback about whether the group is correct.

Use the feedback to improve your next attempts.

Words:
{WORDS}
```

---

### What this tests

Implicit iterative reasoning

* correction ability
* memory via conversation
* heuristic improvement

---

### Weakness

* no structured state
* memory is messy (chat history)
* may repeat mistakes
* hard to audit

---

## External Semantic State 

### Definition

Explicit structured state outside the model

* model does not rely on chat history
* state is **tracked and updated explicitly**

---

### State

```json
{
  "remaining_words": [],
  "candidate_groups": [],
  "rejected_groups": [],
  "confirmed_groups": []
}
```

---

### Loop

1. model proposes candidate groups
2. system selects or tests one
3. update state:

   * correct → confirmed
   * incorrect → rejected
4. remove words from remaining
5. repeat

---

### Prompt (core idea)

```{markdown}
You are solving a Connections-style puzzle.

You will propose candidate groups based on the current state.

State:
{STATE}

Words:
{WORDS}

Return candidate groups only.
```

---

### What this tests

Explicit, controlled state

* consistency
* no repeated mistakes (if implemented correctly)
* auditability

---

### Strength

* clear separation:

  * reasoning vs memory
* inspectable
* controllable

---

### Risk

* bad state design → worse performance
* early mistakes may propagate

---